# 03 · Distillation Validation

This notebook validates the LLM-distilled racquet descriptions before they enter the
search pipeline. The distillation step (`distill.py`) compresses Tennis Warehouse's
long marketing copy into short, spec-focused summaries used for semantic embedding.

Because these summaries are LLM-generated, they should not be trusted blindly. This
notebook runs a series of structural and content checks to confirm the distillation
output is complete, aligned to the cleaned data, and qualitatively reasonable before
it is embedded and served.

**Checks performed**
1. Structural integrity — row counts and `racquet_id` alignment between cleaned specs and distilled descriptions
2. Completeness — no empty or missing distilled descriptions
3. Length sanity — distilled text is meaningfully shorter than the source, but not degenerate
4. Qualitative spot-check — manual side-by-side review of a random sample


## Setup

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)


In [2]:
# Point these at the current interim artifacts.
DATA_DIR = Path.cwd().parent / "data"
INTERIM = DATA_DIR / "interim"

CLEANED_PATH = INTERIM / "racquets_cleaned_2026_06_24.csv"
DISTILLED_PATH = INTERIM / "distilled_descriptions_2026_06_24.csv"


In [3]:
cleaned_df = pd.read_csv(CLEANED_PATH)
distilled_df = pd.read_csv(DISTILLED_PATH)

print(f"Cleaned:   {len(cleaned_df)} rows")
print(f"Distilled: {len(distilled_df)} rows")


Cleaned:   333 rows
Distilled: 333 rows


## 1 · Structural integrity

The distilled descriptions must correspond exactly to the cleaned racquets — one
distilled description per racquet, keyed by `racquet_id`, with no rows added or
dropped. Set equality alone is insufficient: a duplicate `racquet_id` within one
frame would pass a set check but break the one-to-one alignment the pipeline assumes.
So we check both row count parity and set equality of IDs.


In [4]:
cleaned_ids = cleaned_df["racquet_id"]
distilled_ids = distilled_df["racquet_id"]

same_length = len(cleaned_ids) == len(distilled_ids)
same_id_set = set(cleaned_ids) == set(distilled_ids)
no_dupe_cleaned = cleaned_ids.is_unique
no_dupe_distilled = distilled_ids.is_unique

print(f"Equal row counts:        {same_length}")
print(f"Identical id sets:       {same_id_set}")
print(f"No duplicate ids (clean): {no_dupe_cleaned}")
print(f"No duplicate ids (dist):  {no_dupe_distilled}")

assert same_length, "Row count mismatch between cleaned and distilled data"
assert same_id_set, "racquet_id sets differ between cleaned and distilled data"
assert no_dupe_cleaned and no_dupe_distilled, "Duplicate racquet_id detected"
print("\nStructural integrity checks passed.")


Equal row counts:        True
Identical id sets:       True
No duplicate ids (clean): True
No duplicate ids (dist):  True

Structural integrity checks passed.


We merge on `racquet_id` with `validate="one_to_one"` so pandas itself enforces the
one-to-one relationship. A merge that drops or multiplies rows will raise here rather
than silently producing misaligned data downstream.


In [5]:
merged = cleaned_df.merge(
    distilled_df, on="racquet_id", how="inner", validate="one_to_one"
)

assert len(merged) == len(cleaned_df), (
    f"Merge changed row count: {len(cleaned_df)} -> {len(merged)}"
)
print(f"Merged cleanly: {len(merged)} rows")


Merged cleanly: 333 rows


## 2 · Completeness

Every racquet must have a non-empty distilled description. An empty or whitespace-only
summary would produce a meaningless embedding and pollute semantic search results.


In [6]:
desc = merged["distilled_description"].fillna("").str.strip()

n_missing = merged["distilled_description"].isna().sum()
n_empty = (desc == "").sum()

print(f"Null distilled descriptions:  {n_missing}")
print(f"Empty distilled descriptions: {n_empty}")

if n_missing or n_empty:
    display(merged.loc[desc == "", ["racquet_id", "racquet_name"]])

assert n_missing == 0, "Found null distilled descriptions"
assert n_empty == 0, "Found empty distilled descriptions"
print("\nCompleteness checks passed.")


Null distilled descriptions:  0
Empty distilled descriptions: 0

Completeness checks passed.


## 3 · Length sanity

Distillation should compress the source copy — the distilled text should be
meaningfully shorter than the original marketing description. At the same time, a
distilled description that is *too* short (e.g. a single truncated phrase) likely
indicates a failed or degenerate generation. We look at the distribution of both
raw and distilled lengths and the compression ratio, then flag outliers.


In [7]:
merged["orig_len"] = merged["racquet_description"].fillna("").str.len()
merged["dist_len"] = merged["distilled_description"].fillna("").str.len()
merged["compression_ratio"] = merged["dist_len"] / merged["orig_len"].replace(0, pd.NA)

length_summary = merged[["orig_len", "dist_len", "compression_ratio"]].describe()
length_summary


,orig_len,dist_len,compression_ratio
count,333.000000,333.000000,333.000000
mean,1212.300300,305.162162,0.260502
std,225.258874,45.715570,0.062892
min,549.000000,167.000000,0.134513
25%,1070.000000,278.000000,0.219147
50%,1209.000000,301.000000,0.247359
75%,1372.000000,335.000000,0.291542
max,1810.000000,444.000000,0.497207


In [8]:
# Flag suspiciously short distilled descriptions for manual review.
SHORT_THRESHOLD = 80  # characters

short = merged[merged["dist_len"] < SHORT_THRESHOLD]
print(f"{len(short)} distilled descriptions under {SHORT_THRESHOLD} characters")
short[["racquet_id", "racquet_name", "dist_len", "distilled_description"]]


0 distilled descriptions under 80 characters


,racquet_id,racquet_name,dist_len,distilled_description


In [9]:
# Flag any distilled descriptions that are LONGER than the source — this should
# essentially never happen and would indicate the distillation did not compress.
expanded = merged[merged["dist_len"] > merged["orig_len"]]
print(f"{len(expanded)} distilled descriptions longer than their source")
expanded[["racquet_id", "racquet_name", "orig_len", "dist_len"]]


0 distilled descriptions longer than their source


,racquet_id,racquet_name,orig_len,dist_len


## 4 · Qualitative spot-check

Structural checks confirm the data is *complete and aligned*, but not that the
distillations are *good*. This final step is a manual side-by-side read of a random
sample: does the distilled text preserve the racquet's key characteristics (weight,
stiffness, stroke style, intended player) while dropping marketing fluff?

A fixed random seed keeps the sample reproducible across runs.


In [10]:
sample = merged.sample(10, random_state=42)

for _, row in sample.iterrows():
    print("=" * 80)
    print(f"{row['racquet_id']}  ·  {row['racquet_name']}")
    print("-" * 80)
    print("ORIGINAL:")
    print(row["racquet_description"])
    print()
    print("DISTILLED:")
    print(row["distilled_description"])
    print()


PS1020  ·  Babolat Pure Strike 100 16x20 Carbon Grey
--------------------------------------------------------------------------------
ORIGINAL:
Babolat adds a sleek carbon grey paint job to the Pure Strike 100 16x20 Babolat adds a carbon grey cosmetic to the Pure Strike 100 16x20! This racquet not only delivers more control than the 100 16x19, but it also packs enough pop to close out points. With its predictable trajectory and zippy 320 swingweight, this stick offers easy targeting from the baseline and quick handling at net. It also packs a sub-64 RA stiffness, giving it an arm-friendly feel with outstanding ball feedback. As with other members of the family, this racquet has Control Frame Technology , which blends square and elliptical cross sections to deliver a stable hitting experience with excellent ball feedback. To help with vibration dampening, Babolat adds NF2-Tech (flax fibers) to both the handle and hoop. The 100 16x20 Carbon Grey also gives you the benefit of FSI Control 

## Summary

If all assertions above passed and the spot-checked samples read sensibly, the
distilled descriptions are cleared to move into the embedding and BM25 indexing
steps (`build_processed_artifacts.py`).

Any failures here should be traced back to `distill.py` — either a generation failure
for specific racquets, or an ID-alignment issue between the cleaning and distillation
stages — and resolved before rebuilding the serving artifacts.
